# AWS Bedrock + LiteLLM Testing

Simple tests for Bedrock models via LiteLLM with cost tracking.

In [1]:
import os

from dotenv import load_dotenv
from litellm import completion, completion_cost

load_dotenv()
print("✓ Loaded")

✓ Loaded


In [2]:
# Config
region = os.getenv("AWS_REGION")
token = os.getenv("AWS_BEARER_TOKEN_BEDROCK")
llama_1b = os.getenv("BEDROCK_LLAMA_1B")
llama_3b = os.getenv("BEDROCK_LLAMA_3B")
llama_8b = os.getenv("BEDROCK_LLAMA_8B")

print(f"Region: {region}")
print(f"Token: {'✓' if token else '✗'}")
print(f"Models: {llama_1b}, {llama_3b}, {llama_8b}")

Region: us-east-1
Token: ✓
Models: us.meta.llama3-2-1b-instruct-v1:0, us.meta.llama3-2-3b-instruct-v1:0, us.meta.llama3-1-8b-instruct-v1:0


## Test 1: Basic Call

In [3]:
response = completion(
    model=f"bedrock/{llama_1b}",
    messages=[{"role": "user", "content": "Say hello in 5 words"}],
    max_tokens=50,
    temperature=0,
)

print(f"Response: {response.choices[0].message.content}")
print(f"Tokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")

Response: Hello, it's nice to meet you.
Tokens: 51
Cost: $0.000005


## Test 2: Math Reasoning

In [4]:
response = completion(
    model=f"bedrock/{llama_8b}",
    messages=[
        {"role": "system", "content": "Think step by step"},
        {
            "role": "user",
            "content": "If I have 3 apples and buy 2 more, then give 1 away, how many?",
        },
    ],
    max_tokens=150,
    temperature=0,
)

print(response.choices[0].message.content)
print(f"\nTokens: {response.usage.total_tokens}")
print(f"Cost: ${completion_cost(completion_response=response):.6f}")



Let's break it down step by step:

1. You start with 3 apples.
2. You buy 2 more apples, so now you have 3 + 2 = 5 apples.
3. You give 1 apple away, so now you have 5 - 1 = 4 apples.

So, you have 4 apples.

Tokens: 114
Cost: $0.000025


## Test 3: Compare All Models

In [5]:
question = "What is 2+2?"
total_cost = 0

for name, model in [("1B", llama_1b), ("3B", llama_3b), ("8B", llama_8b)]:
    response = completion(
        model=f"bedrock/{model}",
        messages=[{"role": "user", "content": question}],
        max_tokens=20,
        temperature=0,
    )

    cost = completion_cost(completion_response=response)
    total_cost += cost

    print(f"{name}: {response.choices[0].message.content}")
    print(f"     Tokens: {response.usage.total_tokens}, Cost: ${cost:.6f}\n")

print(f"Total: ${total_cost:.6f}")

1B: 2 + 2 = 4.
     Tokens: 51, Cost: $0.000005

3B: 2 + 2 = 4.
     Tokens: 51, Cost: $0.000008

8B: 

2 + 2 = 4
     Tokens: 31, Cost: $0.000007

Total: $0.000020


## Test 5: Temperature Effects

In [6]:
prompt = "The future of AI is"

for temp in [0.0, 0.7, 1.0]:
    response = completion(
        model=f"bedrock/{llama_3b}",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=30,
        temperature=temp,
    )

    print(f"Temp {temp}: {response.choices[0].message.content}\n")

Temp 0.0: The future of AI is a topic of much debate and speculation. Here are some potential trends and developments that could shape the future of AI:

1.

Temp 0.7: The future of AI is a topic of much speculation and debate. Based on current trends and advancements, here are some potential directions the future of AI might

Temp 1.0: The future of AI is a topic of ongoing debate and exploration among experts, researchers, and industry leaders. Based on current trends and advancements, here are



## Test 6: DSPy with LiteLLM

In [10]:
import dspy
from dspy import LM

# Configure DSPy to use LiteLLM with Bedrock
lm = LM(model=f"bedrock/{llama_1b}", temperature=0, max_tokens=100)
dspy.configure(lm=lm)


# Define a simple signature
class BasicQA(dspy.Signature):
    """Answer questions concisely."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


# Create and test predictor
qa = dspy.ChainOfThought(BasicQA)
question = "What is the capital of France?"
response = qa(question=question)

print(f"Question: {question}")
print(f"Answer: {response.answer}")

# Inspect available fields
print(f"\nAvailable fields: {list(response.keys())}")

# Try to access reasoning if available
if hasattr(response, "reasoning"):
    print(f"\nReasoning: {response.reasoning}")
elif "reasoning" in response:
    print(f"\nReasoning: {response.reasoning}")

Question: What is the capital of France?
Answer: Paris

Available fields: ['reasoning', 'answer']

Reasoning: The capital of France is Paris.


In [14]:
# Access DSPy call history for cost tracking
from litellm import completion_cost

# Get the last call from DSPy's history
last_call = lm.history[-1]

print("DSPy Call History:")
print(f"  Prompt tokens: {last_call['usage']['prompt_tokens']}")
print(f"  Completion tokens: {last_call['usage']['completion_tokens']}")
print(f"  Total tokens: {last_call['usage']['total_tokens']}")

# Calculate cost from the response
if "response" in last_call:
    cost = completion_cost(completion_response=last_call["response"])
    print(f"  Cost: ${cost:.6f}")
else:
    print("\n  Note: Raw response not available in history")
    print("  You can enable this with: lm = LM(..., cache=False)")

DSPy Call History:
  Prompt tokens: 192
  Completion tokens: 77
  Total tokens: 269
  Cost: $0.001904


Generative AI was used to assist in building this notebook.